# Extração dos Dados

In [23]:
!pip install -q sidrapy
import sidrapy
import pandas as pd

##Função de extração via API

In [24]:
def extrair_sidra(tabela, variavel, classificacoes=None):
    params = {
        "table_code": tabela,
        "territorial_level": "6", #municipios
        "ibge_territorial_code": "all",
        "variable": variavel
    }

    if classificacoes:
        params["classifications"] = classificacoes

    df_site = sidrapy.get_table(**params)

    #ajustes na nomenclatura e limpeza
    df = df_site.iloc[1:].copy()
    df = df.rename(columns={'V': 'Valor', 'D1C': 'Cod_Municipio', 'D1N': 'Nome_Municipio'})
    df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')

    return df[['Cod_Municipio', 'Nome_Municipio', 'Valor']]


##Dados da escolaridade

In [25]:
df_esc = extrair_sidra(tabela="10062", variavel="all")
if df_esc is not None:
  df_escolaridade = df_esc.rename(columns={'Valor': 'Anos_Estudo_Medio'})[['Cod_Municipio', 'Nome_Municipio', 'Anos_Estudo_Medio']]
  print('Escolaridade extraída com sucesso')

else:
  print('Erro na extração da escolaridade')


Escolaridade extraída com sucesso


## Dados de Renda

Para essa tabela é nescessário usar uma classificação:

- "2": "6794" $\rightarrow$ O filtro 2 é o Sexo, e o código 6794 significa Total.
- "11913": "96165" $\rightarrow$ O filtro 11913 é a Posição na Ocupação, e o código 96165 significa Total.

In [26]:
class_renda = {"2": "6794", "11913": "96165"}
df_rnd = extrair_sidra(tabela="10280", variavel="13536", classificacoes=class_renda)
if df_rnd is not None:
  df_renda = df_rnd.rename(columns={'Valor': 'Renda_Media_Mensal'})[['Cod_Municipio', 'Renda_Media_Mensal']]
  print('Renda extraída com sucesso')

else:
  print('Erro na extração da renda')


Renda extraída com sucesso


## Dados referentes à idade
A Tabela 9515 tem três indicadores demográficos diferentes para cada município:

- O Índice de envelhecimento

- A Idade mediana da população

- A Razão de sexo

Nesse ordem, então para obtermos apenas a linha da idade mediana é nescessário retirar as outras 2 usando o `groupby` no índice do meio (`nth(1)`)

In [27]:
df_idg = extrair_sidra(tabela="9515", variavel="all")
if df_idg is not None and not df_idg.empty:
  df_idade = df_idg.groupby('Cod_Municipio').nth(1).reset_index()
  df_idade = df_idade.rename(columns={'Valor': 'Idade_Mediana'})[['Cod_Municipio', 'Idade_Mediana']]
  print('Idade extraída com sucesso')

else:
  print('Erro na extração da idade')


Idade extraída com sucesso


## Dados referentes à informalidade

A classificação segue a mesma lógica:
- "2": "6794" $\rightarrow$ O filtro 2 é o Sexo, e o código 6794 significa Total.
- "86": "95251" $\rightarrow$ O filtro 86 é a Cor ou raça, e o código 95251 significa Total.

A taxa de informalidade não é uma dado informado diretamente pelo censo, por isso é nescessário extrair todos os dados nescessários para o seu cálculo:

$$\text{Informalidade} = \frac{\text{Empregados sem CTPS} + \text{Domésticos sem CTPS} + \text{Conta própria sem CNPJ} + \text{Trabalho familiar}}{\text{Total de Ocupados}}$$

CTPS é Carteira de Trabalho e Previdência Social.

Os índices usando no código representam essas populações:

- "96165": Total de Pessoas Ocupadas

- "31723":  Empregados no setor privado sem carteira de trabalho assinada

- "79368": Trabalhadores domésticos sem carteira de trabalho assinada

- "45937": Trabalhadores por conta própria sem CNPJ

- "31731": Trabalhadores familiares auxiliares

In [30]:
class_base = {"2": "6794", "86": "95251"}

df_tot = extrair_sidra("10261", "4090", {**class_base, "11913": "96165"})
df_emp_sc = extrair_sidra("10261", "4090", {**class_base, "11913": "31723"})
df_dom_sc = extrair_sidra("10261", "4090", {**class_base, "11913": "79368"})
df_cp_sc = extrair_sidra("10261", "4090", {**class_base, "11913": "45937"})
df_tf = extrair_sidra("10261", "4090", {**class_base, "11913": "31731"})

if df_tot is not None and not df_tot.empty:
  df_trab = df_tot[['Cod_Municipio', 'Nome_Municipio']].copy()
  df_trab['Total_Ocupados'] = df_tot['Valor'].values

  df_trab['Numerador_Informais'] = (
    df_emp_sc['Valor'].fillna(0).values +
    df_dom_sc['Valor'].fillna(0).values +
    df_cp_sc['Valor'].fillna(0).values +
    df_tf['Valor'].fillna(0).values
  )

  df_trab['Taxa_Informalidade'] = df_trab['Numerador_Informais'] / df_trab['Total_Ocupados']
  print('Informalidade extraída com sucesso')

else:
  print('Erro na extração da informalidade')

Informalidade extraída com sucesso


## União das bases de dados

Unimos as tabelas usando `.merge()` e retiramos as colunas duplicadas criadas por esse método

In [ ]:
df_final = df_escolaridade.merge(df_renda, on='Cod_Municipio', how='outer')
df_final = df_final.merge(df_idade, on='Cod_Municipio', how='outer')
df_final = df_final.merge(df_trab[['Cod_Municipio', 'Nome_Municipio', 'Taxa_Informalidade']], on='Cod_Municipio', how='outer')

if 'Nome_Municipio_y' in df_final.columns:
  df_final = df_final.rename(columns={'Nome_Municipio_x': 'Nome_Municipio'})
  df_final = df_final.drop(columns=['Nome_Municipio_y'])

  arquivo = "Dados_IBGE.csv"
  df_final.to_csv(arquivo, index=False)
  print(f"Tabela final salva em '{arquivo}'.")

Tabela final salva em 'Dados_IBGE.xlsx'.


In [34]:
df_final.head(10)

,Cod_Municipio,Nome_Municipio,Anos_Estudo_Medio,Renda_Media_Mensal,Idade_Mediana,Taxa_Informalidade
0,1100015,Alta Floresta D'Oeste - RO,7.1,2154.81,34.0,0.553404
1,1100023,Ariquemes - RO,8.9,2569.41,32.0,0.406944
2,1100031,Cabixi - RO,7.3,2281.93,38.0,0.512653
3,1100049,Cacoal - RO,9.1,2781.78,33.0,0.359134
4,1100056,Cerejeiras - RO,8.6,2741.65,34.0,0.393130
5,1100064,Colorado do Oeste - RO,8.6,2753.72,36.0,0.406672
6,1100072,Corumbiara - RO,7.5,2202.29,35.0,0.541292
7,1100080,Costa Marques - RO,8.1,2001.88,30.0,0.614356
8,1100098,Espigão D'Oeste - RO,7.9,2389.36,33.0,0.483034
9,1100106,Guajará-Mirim - RO,8.8,2261.61,29.0,0.457865
